In [12]:
import os
import pandas as pd
import json

# Inicializar df_merged como un DataFrame vacío para evitar NameError en otras celdas
df_merged = pd.DataFrame()

def load_cohort_data(cohort_path):
    """
    Carga todos los ensayos de una cohorte específica combinando series temporales y metadatos.
    """
    all_trial_dfs = []
    
    # Convertir a path absoluto para evitar problemas con el directorio de trabajo del notebook
    abs_cohort_path = os.path.abspath(cohort_path)
    print(f"Buscando datos en: {abs_cohort_path}")
    
    if not os.path.exists(abs_cohort_path):
        print(f" Error: El path {abs_cohort_path} no existe.")
        return pd.DataFrame()

    patients = [f for f in os.listdir(abs_cohort_path) if os.path.isdir(os.path.join(abs_cohort_path, f))]
    if not patients:
        print(f" No se encontraron subcarpetas de pacientes en {abs_cohort_path}")
        return pd.DataFrame()
        
    for patient_folder in patients:
        patient_path = os.path.join(abs_cohort_path, patient_folder)
            
        # Recorrer cada ensayo dentro de la carpeta del paciente
        for trial_folder in os.listdir(patient_path):
            trial_path = os.path.join(patient_path, trial_folder)
            if not os.path.isdir(trial_path):
                continue
                
            # Identificar archivos relevantes
            processed_file = None
            meta_file = None
            
            for file in os.listdir(trial_path):
                if "processed" in file and file.endswith(".txt"):
                    processed_file = os.path.join(trial_path, file)
                elif "meta" in file and file.endswith(".json"):
                    meta_file = os.path.join(trial_path, file)
            
            if processed_file and meta_file:
                # Cargar datos de sensores procesados
                df_trial = pd.read_csv(processed_file, sep='\t')
                
                # Cargar metadatos del ensayo/sujeto
                with open(meta_file, 'r', encoding='utf-8') as f:
                    metadata = json.load(f)
                
                # Integrar metadatos escalares
                for key, value in metadata.items():
                    if not isinstance(value, list):
                        df_trial[key] = value
                
                # Añadir identificador único de ensayo
                df_trial['trial_id'] = trial_folder
                all_trial_dfs.append(df_trial)
    
    if all_trial_dfs:
        return pd.concat(all_trial_dfs, ignore_index=True)
    return pd.DataFrame()

# Definición de rutas relativas (subiendo un nivel desde la carpeta 'quick_start')
cipn_path = os.path.join("..", "data", "neuro", "CIPN")
healthy_path = os.path.join("..", "data", "healthy", "HS")

# Ejecutar la carga de ambas cohortes
df_cipn = load_cohort_data(cipn_path)
df_healthy = load_cohort_data(healthy_path)

# Combinar los datos
if not df_cipn.empty or not df_healthy.empty:
    df_merged = pd.concat([df_cipn, df_healthy], ignore_index=True)
    
    print(f"\nConsolidación completada correctamente.")
    print(f" - Registros CIPN: {len(df_cipn)}")
    print(f" - Registros Healthy: {len(df_healthy)}")
    print(f" - Total final: {len(df_merged)} filas, {df_merged.shape[1]} columnas")
    print(f" - Sujetos únicos: {df_merged['subject'].nunique()}")
    
    display(df_merged.head())
else:
    print("\n ERROR: No se pudieron cargar datos. Verifica que las carpetas 'data/neuro/CIPN' y 'data/healthy/HS' existan.")

Buscando datos en: c:\Users\cvill\Desktop\dataset\dataset\data\neuro\CIPN
Buscando datos en: c:\Users\cvill\Desktop\dataset\dataset\data\healthy\HS

Consolidación completada correctamente.
 - Registros CIPN: 295468
 - Registros Healthy: 804347
 - Total final: 1099815 filas, 59 columnas
 - Sujetos únicos: 92


,PacketCounter,HE_Acc_X,HE_Acc_Y,HE_Acc_Z,HE_FreeAcc_X,HE_FreeAcc_Y,HE_FreeAcc_Z,HE_Gyr_X,HE_Gyr_Y,HE_Gyr_Z,LB_Acc_X,LB_Acc_Y,LB_Acc_Z,LB_FreeAcc_X,LB_FreeAcc_Y,LB_FreeAcc_Z,LB_Gyr_X,LB_Gyr_Y,LB_Gyr_Z,LF_Acc_X,LF_Acc_Y,LF_Acc_Z,LF_FreeAcc_X,LF_FreeAcc_Y,LF_FreeAcc_Z,LF_Gyr_X,LF_Gyr_Y,LF_Gyr_Z,RF_Acc_X,RF_Acc_Y,RF_Acc_Z,RF_FreeAcc_X,RF_FreeAcc_Y,RF_FreeAcc_Z,RF_Gyr_X,RF_Gyr_Y,RF_Gyr_Z,subject,age,gender,height,weight,BMI,laterality,group,pathology,pathologyKey,clinicalDeficitSide,evaluationScoreName,evaluationScoreValue,session,daysSinceFirstSession,trial,freq,sensor,protocol,TUG,visualGaitAssessment,trial_id
0,0.0,8.851556,0.038272,3.851430,-0.012815,-0.085385,0.060542,-0.017605,-0.042035,0.002398,9.695210,0.201117,0.727871,0.019864,-0.130039,-0.242191,-0.000073,-0.010679,-0.001345,5.885995,-1.368492,7.740096,0.017344,-0.006433,-0.005292,0.001327,0.011367,-0.005532,6.749280,1.611784,7.065155,-0.015642,0.003120,-0.057700,0.008196,-0.012421,-0.005756,CIPN_1,70,F,1.65,72.0,26.4,right,neuro,toxic peripheral neuropathy,CIPN,None,TNSc (/28),6.0,1,0,1,100.0,TechnoConcept,10.0m - uturn - 10.0m,Not evaluated,1.0,CIPN_1_1
1,1.0,8.852785,0.038893,3.846106,-0.014044,-0.086006,0.055218,-0.017579,-0.044085,0.002425,9.695554,0.170534,0.789709,0.019519,-0.099456,-0.180353,0.001201,-0.006462,0.000191,5.874449,-1.367401,7.747981,0.005798,-0.005342,0.002594,0.002822,0.006509,-0.005217,6.757378,1.619507,7.097594,-0.007544,0.010844,-0.025260,0.009258,-0.010667,-0.003609,CIPN_1,70,F,1.65,72.0,26.4,right,neuro,toxic peripheral neuropathy,CIPN,None,TNSc (/28),6.0,1,0,1,100.0,TechnoConcept,10.0m - uturn - 10.0m,Not evaluated,1.0,CIPN_1_1
2,2.0,8.855059,0.040143,3.838512,-0.016317,-0.087255,0.047624,-0.018410,-0.045842,0.005294,9.697427,0.143324,0.839765,0.017646,-0.072246,-0.130297,0.001781,-0.002684,0.001257,5.865677,-1.366374,7.754034,-0.002974,-0.004314,0.008647,0.004572,0.002867,-0.005102,6.763702,1.624395,7.121677,-0.001220,0.015732,-0.001178,0.009453,-0.009515,-0.001342,CIPN_1,70,F,1.65,72.0,26.4,right,neuro,toxic peripheral neuropathy,CIPN,None,TNSc (/28),6.0,1,0,1,100.0,TechnoConcept,10.0m - uturn - 10.0m,Not evaluated,1.0,CIPN_1_1
3,3.0,8.858762,0.042189,3.827933,-0.020021,-0.089302,0.037045,-0.020498,-0.047002,0.012395,9.701371,0.121910,0.870776,0.013702,-0.050832,-0.099286,0.001182,0.000429,0.001637,5.861289,-1.365340,7.757222,-0.007362,-0.003280,0.011835,0.006385,0.001108,-0.005136,6.767233,1.625208,7.132829,0.002310,0.016544,0.009974,0.008506,-0.009136,0.000853,CIPN_1,70,F,1.65,72.0,26.4,right,neuro,toxic peripheral neuropathy,CIPN,None,TNSc (/28),6.0,1,0,1,100.0,TechnoConcept,10.0m - uturn - 10.0m,Not evaluated,1.0,CIPN_1_1
4,4.0,8.863383,0.044540,3.815668,-0.024642,-0.091653,0.024780,-0.023561,-0.047341,0.022870,9.706638,0.107103,0.882457,0.008435,-0.036025,-0.087604,-0.000678,0.002886,0.001472,5.861181,-1.364092,7.757677,-0.007470,-0.002033,0.012289,0.007602,0.001073,-0.004950,6.768073,1.622863,7.131966,0.003151,0.014200,0.009111,0.006927,-0.009144,0.002397,CIPN_1,70,F,1.65,72.0,26.4,right,neuro,toxic peripheral neuropathy,CIPN,None,TNSc (/28),6.0,1,0,1,100.0,TechnoConcept,10.0m - uturn - 10.0m,Not evaluated,1.0,CIPN_1_1


In [13]:
pd.set_option('display.max_columns', None)

if not df_merged.empty:
    print("Patologías detectadas:")
    print(df_merged['pathology'].unique())
else:
    print("El DataFrame df_merged está vacío. Ejecuta la celda superior primero.")

Patologías detectadas:
<StringArray>
['toxic peripheral neuropathy', 'healthy subject']
Length: 2, dtype: str


In [14]:
if not df_merged.empty:
    filename = 'datos_completos_neuro_healthy.csv'
    df_merged.to_csv(filename, sep="|", index=False)
    print(f"Archivo '{filename}' guardado correctamente (tamaño: {len(df_merged)} filas).")
else:
    print("No hay datos para guardar.")

Archivo 'datos_completos_neuro_healthy.csv' guardado correctamente (tamaño: 1099815 filas).


In [15]:
df_merged.shape

(1099815, 59)